# 01 — Data overview

**Child Mind Institute — Problematic Internet Use**

Цель ноутбука — зафиксировать базовую структуру табличных данных до полноценного EDA:

- размеры `train` и `test`;
- типы данных;
- уникальность `id`;
- различия между наборами колонок;
- доступные model features;
- target-related / leakage-признаки;
- структуру `data_dictionary.csv`.

Actigraphy/parquet данные здесь намеренно не используются.


## 1. Imports and configuration

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

TARGET = "sii"
ID_COL = "id"


## 2. Paths and data loading

In [5]:
# Работает при запуске Jupyter как из корня проекта, так и из project/notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
DATA_DICT_PATH = DATA_DIR / "data_dictionary.csv"

for path in [TRAIN_PATH, TEST_PATH, DATA_DICT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")

print("Project root:", PROJECT_ROOT.resolve())
print("Data directory:", DATA_DIR.resolve())


Project root: C:\Users\Honor\predict-internet-usage-ivanov-secret
Data directory: C:\Users\Honor\predict-internet-usage-ivanov-secret\data


In [6]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
data_dict = pd.read_csv(DATA_DICT_PATH)

print("Train:", train.shape)
print("Test:", test.shape)
print("Data dictionary:", data_dict.shape)

display(train.head())


Train: (3960, 82)
Test: (20, 59)
Data dictionary: (81, 6)


,id,Basic_Demos-Enroll_Season,Basic_Demos-Age,Basic_Demos-Sex,CGAS-Season,CGAS-CGAS_Score,Physical-Season,Physical-BMI,Physical-Height,Physical-Weight,Physical-Waist_Circumference,Physical-Diastolic_BP,Physical-HeartRate,Physical-Systolic_BP,Fitness_Endurance-Season,Fitness_Endurance-Max_Stage,Fitness_Endurance-Time_Mins,Fitness_Endurance-Time_Sec,FGC-Season,FGC-FGC_CU,FGC-FGC_CU_Zone,FGC-FGC_GSND,FGC-FGC_GSND_Zone,FGC-FGC_GSD,FGC-FGC_GSD_Zone,FGC-FGC_PU,FGC-FGC_PU_Zone,FGC-FGC_SRL,FGC-FGC_SRL_Zone,FGC-FGC_SRR,FGC-FGC_SRR_Zone,FGC-FGC_TL,FGC-FGC_TL_Zone,BIA-Season,BIA-BIA_Activity_Level_num,BIA-BIA_BMC,BIA-BIA_BMI,BIA-BIA_BMR,BIA-BIA_DEE,BIA-BIA_ECW,BIA-BIA_FFM,BIA-BIA_FFMI,BIA-BIA_FMI,BIA-BIA_Fat,BIA-BIA_Frame_num,BIA-BIA_ICW,BIA-BIA_LDM,BIA-BIA_LST,BIA-BIA_SMM,BIA-BIA_TBW,PAQ_A-Season,PAQ_A-PAQ_A_Total,PAQ_C-Season,PAQ_C-PAQ_C_Total,PCIAT-Season,PCIAT-PCIAT_01,PCIAT-PCIAT_02,PCIAT-PCIAT_03,PCIAT-PCIAT_04,PCIAT-PCIAT_05,PCIAT-PCIAT_06,PCIAT-PCIAT_07,PCIAT-PCIAT_08,PCIAT-PCIAT_09,PCIAT-PCIAT_10,PCIAT-PCIAT_11,PCIAT-PCIAT_12,PCIAT-PCIAT_13,PCIAT-PCIAT_14,PCIAT-PCIAT_15,PCIAT-PCIAT_16,PCIAT-PCIAT_17,PCIAT-PCIAT_18,PCIAT-PCIAT_19,PCIAT-PCIAT_20,PCIAT-PCIAT_Total,SDS-Season,SDS-SDS_Total_Raw,SDS-SDS_Total_T,PreInt_EduHx-Season,PreInt_EduHx-computerinternet_hoursday,sii
0,00008ff9,Fall,5,0,Winter,51.0,Fall,16.877316,46.0,50.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fall,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,7.0,0.0,6.0,0.0,6.0,1.0,Fall,2.0,2.66855,16.8792,932.498,1492.00,8.25598,41.5862,13.8177,3.06143,9.21377,1.0,24.4349,8.89536,38.9177,19.5413,32.6909,NaN,NaN,NaN,NaN,Fall,5.0,4.0,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,4.0,0.0,4.0,4.0,4.0,4.0,4.0,4.0,2.0,4.0,55.0,NaN,NaN,NaN,Fall,3.0,2.0
1,000fd460,Summer,9,0,NaN,NaN,Fall,14.035590,48.0,46.0,22.0,75.0,70.0,122.0,NaN,NaN,NaN,NaN,Fall,3.0,0.0,NaN,NaN,NaN,NaN,5.0,0.0,11.0,1.0,11.0,1.0,3.0,0.0,Winter,2.0,2.57949,14.0371,936.656,1498.65,6.01993,42.0291,12.8254,1.21172,3.97085,1.0,21.0352,14.97400,39.4497,15.4107,27.0552,NaN,NaN,Fall,2.340,Fall,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Fall,46.0,64.0,Summer,0.0,0.0
2,00105258,Summer,10,1,Fall,71.0,Fall,16.648696,56.5,75.6,NaN,65.0,94.0,117.0,Fall,5.0,7.0,33.0,Fall,20.0,1.0,10.2,1.0,14.7,2.0,7.0,1.0,10.0,1.0,10.0,1.0,5.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Summer,2.170,Fall,5.0,2.0,2.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,2.0,2.0,1.0,1.0,28.0,Fall,38.0,54.0,Summer,2.0,0.0
3,00115b9f,Winter,9,0,Fall,71.0,Summer,18.292347,56.0,81.6,NaN,60.0,97.0,117.0,Summer,6.0,9.0,37.0,Summer,18.0,1.0,NaN,NaN,NaN,NaN,5.0,0.0,7.0,0.0,7.0,0.0,7.0,1.0,Summer,3.0,3.84191,18.2943,1131.430,1923.44,15.59250,62.7757,14.0740,4.22033,18.82430,2.0,30.4041,16.77900,58.9338,26.4798,45.9966,NaN,NaN,Winter,2.451,Summer,4.0,2.0,4.0,0.0,5.0,1.0,0.0,3.0,2.0,2.0,3.0,0.0,3.0,0.0,0.0,3.0,4.0,3.0,4.0,1.0,44.0,Summer,31.0,45.0,Winter,0.0,1.0
4,0016bb22,Spring,18,1,Summer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Summer,1.04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Dataset overview

Проверяем размеры выборок и корректность идентификаторов участников.


In [4]:
overview = pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(train), len(test)],
    "columns": [train.shape[1], test.shape[1]],
    "unique_ids": [train[ID_COL].nunique(), test[ID_COL].nunique()],
    "duplicated_ids": [
        train[ID_COL].duplicated().sum(),
        test[ID_COL].duplicated().sum(),
    ],
})

display(overview)


,dataset,rows,columns,unique_ids,duplicated_ids
0,train,3960,82,3960,0
1,test,20,59,20,0


### Data types

In [5]:
dtype_summary = pd.DataFrame({
    "train": train.dtypes.value_counts(),
    "test": test.dtypes.value_counts(),
}).fillna(0).astype(int)

display(dtype_summary)


,train,test
float64,68,46
object,12,11
int64,2,2


## 4. Train/test schema comparison

Признаки, отсутствующие в `test`, нельзя использовать как обычные model features.

Для дальнейшей работы безопаснее формировать набор входных признаков непосредственно из `test.columns`.


In [6]:
train_only = sorted(set(train.columns) - set(test.columns))
test_only = sorted(set(test.columns) - set(train.columns))
common_cols = sorted(set(train.columns) & set(test.columns))

print(f"Common columns: {len(common_cols)}")
print(f"Train-only columns: {len(train_only)}")
print(f"Test-only columns: {len(test_only)}")

print("\nTrain-only:")
print(train_only)

print("\nTest-only:")
print(test_only)


Common columns: 59
Train-only columns: 23
Test-only columns: 0

Train-only:
['PCIAT-PCIAT_01', 'PCIAT-PCIAT_02', 'PCIAT-PCIAT_03', 'PCIAT-PCIAT_04', 'PCIAT-PCIAT_05', 'PCIAT-PCIAT_06', 'PCIAT-PCIAT_07', 'PCIAT-PCIAT_08', 'PCIAT-PCIAT_09', 'PCIAT-PCIAT_10', 'PCIAT-PCIAT_11', 'PCIAT-PCIAT_12', 'PCIAT-PCIAT_13', 'PCIAT-PCIAT_14', 'PCIAT-PCIAT_15', 'PCIAT-PCIAT_16', 'PCIAT-PCIAT_17', 'PCIAT-PCIAT_18', 'PCIAT-PCIAT_19', 'PCIAT-PCIAT_20', 'PCIAT-PCIAT_Total', 'PCIAT-Season', 'sii']

Test-only:
[]


## 5. Target and leakage-related columns

`PCIAT-PCIAT_Total` используется для получения `sii`, а PCIAT-поля отсутствуют в test.

Поэтому:

- `sii` — target;
- `id` — идентификатор, а не обычный model feature;
- все `PCIAT-*` нужно исключить из model features;
- основной набор model features удобно определять через `test.columns`.


In [7]:
PCIAT_COLS = [c for c in train.columns if c.startswith("PCIAT")]
MODEL_FEATURES = [c for c in test.columns if c != ID_COL]

print("PCIAT columns:", len(PCIAT_COLS))
print("Model features available in test:", len(MODEL_FEATURES))

print("\nPCIAT columns:")
print(PCIAT_COLS)


PCIAT columns: 22
Model features available in test: 58

PCIAT columns:
['PCIAT-Season', 'PCIAT-PCIAT_01', 'PCIAT-PCIAT_02', 'PCIAT-PCIAT_03', 'PCIAT-PCIAT_04', 'PCIAT-PCIAT_05', 'PCIAT-PCIAT_06', 'PCIAT-PCIAT_07', 'PCIAT-PCIAT_08', 'PCIAT-PCIAT_09', 'PCIAT-PCIAT_10', 'PCIAT-PCIAT_11', 'PCIAT-PCIAT_12', 'PCIAT-PCIAT_13', 'PCIAT-PCIAT_14', 'PCIAT-PCIAT_15', 'PCIAT-PCIAT_16', 'PCIAT-PCIAT_17', 'PCIAT-PCIAT_18', 'PCIAT-PCIAT_19', 'PCIAT-PCIAT_20', 'PCIAT-PCIAT_Total']


In [8]:
# Sanity checks
assert TARGET in train.columns, f"{TARGET!r} is absent from train"
assert TARGET not in test.columns, f"{TARGET!r} unexpectedly appears in test"
assert ID_COL in train.columns and ID_COL in test.columns

leakage_in_features = sorted(set(PCIAT_COLS) & set(MODEL_FEATURES))
print("PCIAT columns accidentally present in MODEL_FEATURES:", leakage_in_features)


PCIAT columns accidentally present in MODEL_FEATURES: []


## 6. Data dictionary

In [9]:
display(data_dict.head(20))

print("Dictionary columns:")
print(data_dict.columns.tolist())


,Instrument,Field,Description,Type,Values,Value Labels
0,Identifier,id,Participant's ID,str,NaN,NaN
1,Demographics,Basic_Demos-Enroll_Season,Season of enrollment,str,"Spring, Summer, Fall, Winter",NaN
2,Demographics,Basic_Demos-Age,Age of participant,float,NaN,NaN
3,Demographics,Basic_Demos-Sex,Sex of participant,categorical int,"0,1","0=Male, 1=Female"
4,Children's Global Assessment Scale,CGAS-Season,Season of participation,str,"Spring, Summer, Fall, Winter",NaN
5,Children's Global Assessment Scale,CGAS-CGAS_Score,Children's Global Assessment Scale Score,int,NaN,NaN
6,Physical Measures,Physical-Season,Season of participation,str,"Spring, Summer, Fall, Winter",NaN
7,Physical Measures,Physical-BMI,Body Mass Index (kg/m^2),float,NaN,NaN
8,Physical Measures,Physical-Height,Height (in),float,NaN,NaN
9,Physical Measures,Physical-Weight,Weight (lbs),float,NaN,NaN


Dictionary columns:
['Instrument', 'Field', 'Description', 'Type', 'Values', 'Value Labels']


In [10]:
instrument_summary = (
    data_dict["Instrument"]
    .value_counts(dropna=False)
    .rename_axis("Instrument")
    .to_frame("n_fields")
)

display(instrument_summary)


,n_fields
Instrument,
Parent-Child Internet Addiction Test,22
Bio-electric Impedance Analysis,17
FitnessGram Child,15
Physical Measures,8
FitnessGram Vitals and Treadmill,4
Sleep Disturbance Scale,3
Demographics,3
Children's Global Assessment Scale,2
Physical Activity Questionnaire (Children),2


### Dictionary coverage

Проверяем, какие поля из train описаны в словаре и есть ли записи словаря, отсутствующие в данных.


In [11]:
dictionary_fields = set(data_dict["Field"].dropna())
train_fields = set(train.columns)

not_in_dictionary = sorted(train_fields - dictionary_fields - {ID_COL, TARGET})
dictionary_only = sorted(dictionary_fields - train_fields)

print("Train fields without dictionary entry:", len(not_in_dictionary))
print(not_in_dictionary)

print("\nDictionary fields absent from train:", len(dictionary_only))
print(dictionary_only)


Train fields without dictionary entry: 0
[]

Dictionary fields absent from train: 0
[]
